In [22]:
import os
import faiss
import numpy as np
import openai
from sentence_transformers import SentenceTransformer
from transformers import pipeline

In [3]:
DATA_PATH = "/content/drive/MyDrive/company-policy"
print(os.listdir(DATA_PATH))

['refund policy.txt', 'cancellation policy.txt', 'shipping policy.txt']


In [24]:
def load_documents(folder_path):
    docs = []
    for file in os.listdir(folder_path):
        if file.endswith(".txt"):
            with open(os.path.join(folder_path, file), "r", encoding="utf-8") as f:
                docs.append(f.read())
    return docs
documents = load_documents(DATA_PATH)
print("Documents loaded:", len(documents))

Documents loaded: 3


In [25]:
def chunk_text(text, chunk_size=500, overlap=50):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start:end]))
        start = end - overlap
    return chunks
all_chunks = []
for doc in documents:
    all_chunks.extend(chunk_text(doc))
print("Total chunks:", len(all_chunks))

Total chunks: 3


In [26]:
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded")

Embedding model loaded


In [27]:
embeddings = model.encode(all_chunks)
print("Embeddings shape:", embeddings.shape)

Embeddings shape: (3, 384)


In [28]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))
print("FAISS index ready")

FAISS index ready


In [29]:
def retrieve_chunks(query, k=3):
    query_embedding = model.encode([query])
    distances, indices = index.search(np.array(query_embedding), k)
    return [all_chunks[i] for i in indices[0]]
test_chunks = retrieve_chunks("refund")
print(test_chunks)

['Returns accepted within 30 days of delivery. Items must be unused and in original packaging. Refunds processed to original payment method. Customer pays return shipping unless the item is defective.', 'We kindly ask that you cancel or reschedule at least [24-48] hours before your appointment. Cancellations made within [24-48] hours of the scheduled time may be subject to a fee of [amount/%]. No-shows without prior notification will be charged [amount/%] of the scheduled service cost. Thank you for understanding, as this allows us to offer your spot to another client.', 'A tracking number will be emailed to you once your order has shipped, allowing you to monitor its progress.']


In [30]:
IMPROVED_PROMPT = """
You are a company policy assistant.

Rules:
- Answer ONLY using the provided context.
- If the answer is not found, say:
  "The provided documents do not contain this information."
- Do not guess or add external knowledge.

Context:
{context}

Question:
{question}

Answer in bullet points.
"""

In [31]:
llm = pipeline("text2text-generation", model="google/flan-t5-small")

Device set to use cpu


In [32]:
def ask_llm(prompt):
    result = llm(prompt, max_length=200)
    return result[0]["generated_text"]
print("Policy RAG Assistant Ready")

Policy RAG Assistant Ready


In [ ]:
while True:
    question = input("Ask a question: ")
    if question.lower() == "exit":
        break
    retrieved = retrieve_chunks(question)
    if not retrieved:
        print("The provided documents do not contain this information.")
        continue
    context = "\n\n".join(retrieved)
    prompt = IMPROVED_PROMPT.format(
        context=context,
        question=question
    )
    answer = ask_llm(prompt)
    print("\nAnswer:\n", answer)
    print("*" * 50)




Ask a question: How long does express shipping take?


Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:
 30 days
**************************************************
Ask a question: Can an order be cancelled after delivery?


Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:
 Cancellations made within [24-48] hours of the scheduled time may be subject to a fee of [amount/%]. No-shows without prior notification will be charged [amount/%] of the scheduled service cost. Thank you for understanding, as this allows us to offer your spot to another client. Returns accepted within 30 days of delivery. Items must be unused and in original packaging. Refunds processed to original payment method. Customer pays return shipping unless the item is defective. A tracking number will be emailed to you once your order has shipped, allowing you to monitor its progress.
**************************************************
Ask a question: What is the refund period?


Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:
 30 days of delivery
**************************************************
